process intersection from Osmium pbf

In [37]:
import osmium
import json
import os
from collections import defaultdict
from shapely.geometry import Point, LineString, Polygon
from shapely.strtree import STRtree

# Helper: convert OSM location to Shapely point
def node_point(n):
    return Point(n.location.lon, n.location.lat)

# Collect intersections (same as before)
class IntersectionFinder(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.node_use = defaultdict(set)

    def way(self, w):
        important_highways = {"primary", "secondary", "tertiary"}
        if 'highway' in w.tags and w.tags['highway'] in important_highways:
            for n in w.nodes:
                self.node_use[n.ref].add(w.id)

# Collect railway features
class RailFeatureCollector(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.points = []

    def node(self, n):
        if 'railway' in n.tags and n.tags['railway'] in {'rail', 'level_crossing'}:
            self.points.append(Point(n.location.lon, n.location.lat))

    def way(self, w):
        if 'railway' in w.tags and w.tags['railway'] in {'rail', 'level_crossing'}:
            coords = [(n.lon, n.lat) for n in w.nodes if n.location.valid()]
            if coords:
                self.points.append(LineString(coords).centroid)

# Collect water features
class WaterFeatureCollector(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.points = []

    def node(self, n):
        if 'natural' in n.tags and n.tags['natural'] == 'water':
            self.points.append(Point(n.location.lon, n.location.lat))
        if 'waterway' in n.tags:
            self.points.append(Point(n.location.lon, n.location.lat))

    def way(self, w):
        if 'natural' in w.tags and w.tags['natural'] == 'water':
            coords = [(n.lon, n.lat) for n in w.nodes if n.location.valid()]
            if coords:
                self.points.append(Polygon(coords).centroid)
        if 'waterway' in w.tags:
            coords = [(n.lon, n.lat) for n in w.nodes if n.location.valid()]
            if coords:
                self.points.append(LineString(coords).centroid)

# Final intersection coordinate filter
class IntersectionCoordsExtractor(osmium.SimpleHandler):
    def __init__(self, intersection_ids):
        super().__init__()
        self.intersection_ids = intersection_ids
        self.intersections = []

    def node(self, n):
        if n.id in self.intersection_ids:
            lat = n.location.lat
            lon = n.location.lon

            # Las Vegas bounding box
            if 35.8 <= lat <= 36.4 and -115.5 <= lon <= -114.9:
                self.intersections.append({
                    "id": n.id,
                    "lat": lat,
                    "lon": lon
                })


In [5]:
pbf_path = "Data_collection/nevada-latest.osm.pbf"
output_path = "Data_collection/intersections.json"

In [33]:
finder = IntersectionFinder()
finder.apply_file(pbf_path, locations=False)

intersection_ids = {nid for nid, ways in finder.node_use.items() if len(ways) >= 2}

# Second pass: get coordinates
coords_extractor = IntersectionCoordsExtractor(intersection_ids)
coords_extractor.apply_file(pbf_path)

# Save to JSON
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    json.dump(coords_extractor.intersections, f, indent=2)

print(f"Saved {len(coords_extractor.intersections)} intersections to {output_path}")

Saved 19973 intersections to Data_collection/intersections.json


In [34]:
import json

# Load the full intersection file
with open("Data_collection/intersections.json", "r") as f:
    data = json.load(f)

# Take the first 100 records
subset = data[:100]

# Save to a new file
with open("Data_collection/intersections_subset_100.json", "w") as f:
    json.dump(subset, f, indent=2)

print("✅ Saved first 100 intersections to intersections_subset_100.json")


✅ Saved first 100 intersections to intersections_subset_100.json


In [35]:
import osmium
from collections import defaultdict
import json

class IntersectionCornerExtractor(osmium.SimpleHandler):
    def __init__(self, intersection_ids):
        super().__init__()
        self.intersection_ids = intersection_ids
        self.node_locations = {}
        self.adjacents = defaultdict(set)

    def node(self, n):
        if n.location.valid():
            self.node_locations[n.id] = (n.location.lat, n.location.lon)

    def way(self, w):
        node_ids = [n.ref for n in w.nodes]
        for i, nid in enumerate(node_ids):
            if nid in self.intersection_ids:
                # Add adjacent node(s)
                if i > 0:
                    self.adjacents[nid].add(node_ids[i-1])
                if i < len(node_ids) - 1:
                    self.adjacents[nid].add(node_ids[i+1])

def extract_corners(pbf_path, intersection_ids, output_path):
    handler = IntersectionCornerExtractor(intersection_ids)
    handler.apply_file(pbf_path)

    result = []
    for nid in intersection_ids:
        if nid not in handler.node_locations:
            continue
        center = handler.node_locations[nid]
        corners = [
            handler.node_locations[adj]
            for adj in handler.adjacents[nid]
            if adj in handler.node_locations
        ]
        result.append({
            "id": nid,
            "center": center,
            "corners": corners
        })

    with open(output_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"✅ Saved to {output_path}")


In [36]:
# Assuming you already have a list of intersection IDs
with open("Data_collection/intersections_subset_100.json") as f:
    intersections = json.load(f)

intersection_ids = {i["id"] for i in intersections}

extract_corners(
    pbf_path="Data_collection/nevada-latest.osm.pbf",
    intersection_ids=intersection_ids,
    output_path="Data_collection/intersection_corners.json"
)


✅ Saved to Data_collection/intersection_corners.json


In [3]:
import json

with open("extracted_features/small-nevada-roads.geojson", "r") as f:
    data = json.load(f)

# Loop through features and check for 'maxspeed'
for feature in data["features"]:
    props = feature.get("properties", {})
    if "maxspeed" in props:
        print("Found maxspeed:", props["maxspeed"])
        break
else:
    print("No 'maxspeed' tag found in this file.")

Found maxspeed: 65 mph


In [4]:
import json

with open("extracted_features/small-nevada-roads.geojson", "r") as f:
    data = json.load(f)

found_keys = set()

for feature in data["features"]:
    props = feature.get("properties", {})
    for key in props:
        if key.startswith("parking:lane"):
            found_keys.add(key)

if found_keys:
    print("✅ Found parking-related keys:")
    for key in sorted(found_keys):
        print(f" - {key}")
else:
    print("❌ No parking:lane tags found.")


❌ No parking:lane tags found.


In [10]:
import geopandas as gpd

# Load the GeoJSON file
gdf = gpd.read_file("extracted_features/footpaths.geojson")



# Print width if the column exists
if "width" in gdf.columns:
    print("\nWidth values:")
    print(gdf["width"].dropna().unique())
else:
    print("\nNo 'width' field found in the file.")



Width values:
['5.5' '11' '6' '10' '12' '18' '20' '12.12' '21' "24'" "32'" "8'" "22'"
 "26'" "18'" "20'" '0.5' '.5' '2' '0.3' '14' '24' '9.5' '15' '11.5' '1'
 '3' '22\'8"' '1.5' "40'" "4'" '4\'6"' '26\'0"' '1.2' "12'" '0.8' '0.4'
 '8' "5'" "10'" "2'" "6'" '6\'6"' "3'" '5\'6"' '2\'6"' '1\'6"' "11'" '1.0'
 '0.7' "37'" "49'" "34'" '5' '0.75' '4' '6.5' '7.64' '0.6' '2.5' '3.5' '7'
 '9' '2.4' '3.6' '7.5' '1.8' '1.2446' '1.46' '1.9' '8 feet']


In [1]:
import geopandas as gpd

# Load the GeoJSON file
roads = gpd.read_file("extracted_features/small-nevada-roads.geojson")

# Count each road type
road_counts = roads["highway"].value_counts(dropna=False)

# Display results
print(road_counts)

highway
service           2235
residential       1034
footway            333
primary            231
secondary          228
tertiary            89
path                70
cycleway            23
motorway            21
motorway_link       20
primary_link        20
secondary_link      19
living_street        7
track                4
pedestrian           1
Name: count, dtype: int64


In [3]:
import geopandas as gpd

# Load the input file
roads = gpd.read_file("extracted_features/small-nevada-roads.geojson")

# Define buffer distances in meters (since we'll use a projected CRS)
buffer_map_meters = {
    "motorway": 30,
    "motorway_link": 25,
    "primary": 25,
    "primary_link": 25,
    "secondary": 20,
    "secondary_link": 20,
    "tertiary": 15,
    "residential": 10,
    "living_street": 10,
    "service": 8,
    "pedestrian": 5,
    "track": 5,
    "cycleway": 4,
    "footway": 2.5,
    "path": 2.5
}

# Assign buffer distances
roads["buffer_dist"] = roads["highway"].map(buffer_map_meters).fillna(2)

# Reproject to UTM Zone 11N (meters)
roads_proj = roads.to_crs(epsg=26911)

# Create buffers
roads_proj["geometry"] = roads_proj.geometry.buffer(roads_proj["buffer_dist"])

# Save to GeoJSON (reproject back to WGS84)
roads_buffered = roads_proj.to_crs(epsg=4326)

# Save to new GeoJSON
roads_buffered.to_file("extracted_features/buffered-small-nevada-roads.geojson", driver="GeoJSON")

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import json
from shapely.geometry import Point, LineString, MultiLineString, MultiPoint
from shapely.ops import unary_union, substring
from sklearn.cluster import DBSCAN

# -----------------------------
# PARAMETERS
# -----------------------------
offset = 1.5
lane_buffer_width = 1.5
centerline_buffer_width = 2.0
approach_distance = 30  # meters
signal_buffer_radius = 15
non_signal_buffer_radius = 7

# -----------------------------
# LOAD AND PROJECT DATA
# -----------------------------
roads = gpd.read_file("extracted_features/small-nevada-roads.geojson").to_crs(epsg=26911)
traffic_lights = gpd.read_file("extracted_features/small-traffic_lights.geojson").to_crs(epsg=26911)
intersections = gpd.read_file("extracted_features/small_intersections.geojson").to_crs(epsg=26911)

# -----------------------------
# CREATE INTERSECTION BUFFERS
# -----------------------------
# Cluster traffic lights that are within 15 meters
coords = np.array([[p.x, p.y] for p in traffic_lights.geometry])
db = DBSCAN(eps=15, min_samples=1).fit(coords)
traffic_lights["cluster"] = db.labels_

# Get centroids of each cluster
cluster_centers = (
    traffic_lights
    .dissolve(by="cluster")
    .centroid
    .reset_index(drop=True)
)

# Create 15m buffer for all signalized clusters
signal_buffers = [
    {"geometry": pt.buffer(signal_buffer_radius), "signal": True, "side": "intersection"}
    for pt in cluster_centers
]

# Add 7m buffer for non-signalized intersections that are not near any signal
non_signal_buffers = []
for pt in intersections.geometry:
    if not any(pt.distance(tl) < 10 for tl in traffic_lights.geometry):
        non_signal_buffers.append({
            "geometry": pt.buffer(non_signal_buffer_radius),
            "signal": False,
            "side": "intersection"
        })

# Combine intersection buffers
intersection_buffers = gpd.GeoDataFrame(signal_buffers + non_signal_buffers, crs=roads.crs)

# -----------------------------
# BUFFER ROADS NEAR INTERSECTIONS
# -----------------------------
def cut_line_at_distance(line, distance):
    if line.length <= distance:
        return line, None
    return substring(line, 0, distance), substring(line, distance, line.length)

def safe_parallel_buffers(segment, side):
    results = []
    try:
        segments = list(segment.geoms) if isinstance(segment, MultiLineString) else [segment]
        for seg in segments:
            offset_line = seg.parallel_offset(offset, side)
            if isinstance(offset_line, LineString):
                results.append(offset_line.buffer(lane_buffer_width))
    except Exception:
        pass
    return results

lane_buffers = []
centerline_buffers = []

for _, road in roads.iterrows():
    geom = road.geometry
    if not isinstance(geom, (LineString, MultiLineString)):
        continue

    segments = list(geom.geoms) if isinstance(geom, MultiLineString) else [geom]

    for seg in segments:
        nearby_intersections = intersection_buffers[intersection_buffers.geometry.intersects(seg.buffer(5))]

        if not nearby_intersections.empty:
            for _, inter in nearby_intersections.iterrows():
                inter_point = inter.geometry.centroid
                projected_point = seg.interpolate(seg.project(inter_point))
                trimmed, remainder = cut_line_at_distance(seg, approach_distance)

                if trimmed:
                    lane_buffers.extend([
                        {"geometry": b, "side": "left", "signal": inter["signal"], "road_id": road.name}
                        for b in safe_parallel_buffers(trimmed, "left")
                    ])
                    lane_buffers.extend([
                        {"geometry": b, "side": "right", "signal": inter["signal"], "road_id": road.name}
                        for b in safe_parallel_buffers(trimmed, "right")
                    ])

                if remainder:
                    centerline_buffers.append({
                        "geometry": remainder.buffer(centerline_buffer_width),
                        "side": "center", "signal": inter["signal"], "road_id": road.name
                    })
        else:
            centerline_buffers.append({
                "geometry": seg.buffer(centerline_buffer_width),
                "side": "center", "signal": False, "road_id": road.name
            })

# -----------------------------
# MERGE AND SAVE WITH IDS
# -----------------------------
gdf_lane = gpd.GeoDataFrame(lane_buffers, crs=roads.crs)
gdf_center = gpd.GeoDataFrame(centerline_buffers, crs=roads.crs)
intersection_buffers = intersection_buffers.to_crs(epsg=26911)

# Combine all buffers and assign numeric IDs
gdf_all = pd.concat([
    intersection_buffers,
    gdf_lane.to_crs(epsg=26911),
    gdf_center.to_crs(epsg=26911)
], ignore_index=True).to_crs(epsg=4326)
gdf_all["id"] = list(range(len(gdf_all)))

# Save main buffer file
gdf_all.to_file("extracted_features/road_buffers_near_intersections.geojson", driver="GeoJSON")

# -----------------------------
# BUILD RELATIONSHIP JSON
# -----------------------------
relationships = {}

# Spatial index for efficient lookup
sindex = gdf_all.sindex

for _, row in gdf_all.iterrows():
    row_id = int(row.id)
    row_geom = row.geometry
    row_side = row.side
    row_road = row.get("road_id", None)

    possible_matches_index = list(sindex.intersection(row_geom.bounds))
    possible_matches = gdf_all.iloc[possible_matches_index]

    related_ids = []
    for _, other in possible_matches.iterrows():
        other_id = int(other.id)
        if other_id == row_id:
            continue
        if not row_geom.intersects(other.geometry):
            continue

        # Exclude direct left/right connection from same road
        if row_side in ["left", "right"] and other.side in ["left", "right"] and row_road == other.get("road_id", None):
            continue

        related_ids.append(other_id)

    relationships[row_id] = related_ids

# Save relationship file
with open("extracted_features/buffer_relationships.json", "w") as f:
    json.dump(relationships, f, indent=2)


In [ ]:
import geopandas as gpd

# Load buffered polygons with speed (buffer polygons)
buffers = gpd.read_file("extracted_features/road_buffer_mean_speed_by_hour.geojson").to_crs(epsg=26911)

# Load original roads with highway tag
roads = gpd.read_file("extracted_features/small-nevada-roads.geojson").to_crs(epsg=26911)

# Spatial join: intersect each polygon with road lines
join = gpd.sjoin(buffers, roads[["geometry", "highway"]], how="left", predicate="intersects")

# Determine the most common highway type for each buffer (by ID)
highway_by_id = (
    join.groupby("id")["highway"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    .reset_index()
)

# Merge back into original buffer GeoDataFrame
buffers = buffers.merge(highway_by_id, on="id", how="left")

# Save updated file
buffers.to_crs(epsg=4326).to_file("road_buffer_mean_speed_with_highway.geojson", driver="GeoJSON")


Skipping field mean_speed_list: unsupported OGR type: 3


: 

In [13]:
# compute_slope_json.py
import json
import numpy as np
import rasterio
from affine import Affine

# -------------------------------
# CONFIGURATION
# -------------------------------
INPUT_DEM = "extracted_features/clark_clip.tif"
OUTPUT_JSON = "extracted_features/slope_grid_cropped.json"
STEP = 4                   # Downsample factor
ROUND_DECIMALS = 2         # Decimal places for slope values

# -------------------------------
# SLOPE COMPUTATION
# -------------------------------
def compute_slope_percent(elevation, xres, yres):
    """Compute slope using Horn's method. Returns slope in percent."""
    dzdx = (
        (elevation[0:-2, 2:] + 2 * elevation[1:-1, 2:] + elevation[2:, 2:]) -
        (elevation[0:-2, 0:-2] + 2 * elevation[1:-1, 0:-2] + elevation[2:, 0:-2])
    ) / (8 * xres)

    dzdy = (
        (elevation[2:, 0:-2] + 2 * elevation[2:, 1:-1] + elevation[2:, 2:]) -
        (elevation[0:-2, 0:-2] + 2 * elevation[0:-2, 1:-1] + elevation[0:-2, 2:])
    ) / (8 * yres)

    slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
    slope_percent = np.tan(slope_rad) * 100
    return slope_percent

# -------------------------------
# MAIN SCRIPT
# -------------------------------
def main():
    with rasterio.open(INPUT_DEM) as src:
        elevation = src.read(1).astype("float32")
        transform = src.transform
        nodata = src.nodata

        # Mask nodata
        if nodata is not None:
            elevation = np.where(elevation == nodata, np.nan, elevation)

        xres = transform.a
        yres = -transform.e  # pixel size in y (negative north-up)

        slope = compute_slope_percent(elevation, xres, yres)

        # Pad result to match original shape
        padded = np.full(elevation.shape, np.nan, dtype="float32")
        padded[1:-1, 1:-1] = slope

        slope_ds = padded[::STEP, ::STEP]

        # Round values
        if ROUND_DECIMALS is not None:
            slope_ds = np.round(slope_ds, ROUND_DECIMALS)

        # Prepare JSON output
        a, b, c, d, e, f = transform.to_gdal()
        a_ds = a * STEP
        e_ds = e * STEP

        height_ds, width_ds = slope_ds.shape

        values = [
            [None if not np.isfinite(val) else float(val) for val in row]
            for row in slope_ds
        ]

        meta = {
            "crs": "EPSG:4326",  # or src.crs.to_string() if already WGS84
            "width": width_ds,
            "height": height_ds,
            "transform": [a_ds, b, c, d, e_ds, f],
            "units": "percent",
            "step": STEP,
            "nodata": None if nodata is None else float(nodata)
        }

        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump({"meta": meta, "values": values}, f, ensure_ascii=False, indent=0)

        print(f"✅ Slope exported to {OUTPUT_JSON}")

if __name__ == "__main__":
    main()


✅ Slope exported to extracted_features/slope_grid_cropped.json


In [16]:
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds, bounds as win_bounds
import numpy as np
import math

LATLON_BOUNDS = (-115.10, 36.15, -115.05, 36.20)  # (xmin, ymin, xmax, ymax)
INPUT_TIF = "extracted_features/clark.tif"
OUTPUT_TIF = "extracted_features/clark_clip.tif"

def approx_equal(a, b, tol):
    return all(abs(x - y) <= tol for x, y in zip(a, b))

with rasterio.open(INPUT_TIF) as src:
    # 1) Compute window in source CRS
    utm_bounds_req = transform_bounds("EPSG:4326", src.crs, *LATLON_BOUNDS, densify_pts=21)
    window = from_bounds(*utm_bounds_req, transform=src.transform)

    # 2) Read and write
    transform = src.window_transform(window)
    data = src.read(1, window=window)
    if data.size == 0 or data.shape[0] == 0 or data.shape[1] == 0:
        raise ValueError("❌ Clipping window is empty. Check bounds and CRS.")

    profile = src.profile.copy()
    profile.update({"height": data.shape[0], "width": data.shape[1], "transform": transform})
    with rasterio.open(OUTPUT_TIF, "w", **profile) as dst:
        dst.write(data, 1)

print(f"✅ Clipped DEM saved to {OUTPUT_TIF}")

# -----------------------------
# VERIFICATION BLOCK
# -----------------------------
with rasterio.open(INPUT_TIF) as src:
    utm_bounds_req = transform_bounds("EPSG:4326", src.crs, *LATLON_BOUNDS, densify_pts=21)
    window = from_bounds(*utm_bounds_req, transform=src.transform)
    # a) Bounds of the actual window (in source CRS)
    actual_src_bounds = win_bounds(window, src.transform)  # (left, bottom, right, top)

    # Tolerance = one pixel in each direction
    px_w = abs(src.transform.a)
    px_h = abs(src.transform.e)
    tol_src = max(px_w, px_h)

    # b) The requested UTM bounds vs the snapped window bounds
    if not approx_equal(actual_src_bounds, utm_bounds_req, tol_src):
        raise AssertionError(
            f"❌ Window bounds differ from requested (src CRS) by > 1 pixel.\n"
            f"Requested: {utm_bounds_req}\n"
            f"Actual:    {actual_src_bounds}\n"
            f"Tolerance: {tol_src}"
        )

with rasterio.open(OUTPUT_TIF) as out:
    # c) Output bounds (still in source CRS)
    out_bounds_src = out.bounds  # (left, bottom, right, top) in src.crs
    # d) Compare output bounds (in lat/lon) back to requested LATLON_BOUNDS
    out_bounds_ll = transform_bounds(out.crs, "EPSG:4326", *out_bounds_src, densify_pts=21)

    # Convert tolerance to degrees very roughly via pixel size ratio at this latitude
    # (Safe check uses a 2-pixel margin in degrees using geodesy-lite scale factor.)
    # Horizontal meters/deg ≈ 111320*cos(phi); vertical ≈ 110574
    phi = math.radians((LATLON_BOUNDS[1] + LATLON_BOUNDS[3]) / 2.0)
    m_per_deg_x = 111320 * math.cos(phi)
    m_per_deg_y = 110574
    px_deg_x = (abs(out.transform.a) / m_per_deg_x) if m_per_deg_x > 0 else 1e-6
    px_deg_y = (abs(out.transform.e) / m_per_deg_y) if m_per_deg_y > 0 else 1e-6
    tol_ll = 2 * max(px_deg_x, px_deg_y)  # allow 2 pixels in lon/lat

    if not approx_equal(out_bounds_ll, LATLON_BOUNDS, tol_ll):
        raise AssertionError(
            f"❌ Output geo bounds differ from requested LAT/LON by > ~2 pixels.\n"
            f"Requested (LL): {LATLON_BOUNDS}\n"
            f"Actual (LL):    {tuple(round(v, 8) for v in out_bounds_ll)}\n"
            f"Tolerance deg:  ~{tol_ll:.8f}"
        )

    # e) Verify not all NoData
    nodata = out.nodata
    arr = out.read(1)
    if nodata is not None:
        valid_mask = arr != nodata
    else:
        valid_mask = ~np.isnan(arr)
    valid_ratio = float(np.count_nonzero(valid_mask)) / arr.size

    if valid_ratio == 0.0:
        raise AssertionError("❌ Clipped raster is entirely NoData. Check bounds alignment.")
    print(f"✅ Bounds check passed. Valid pixels: {valid_ratio:.2%}")

print("🎯 Verification complete: window snapped correctly, geobounds match requested area, and data is valid.")


✅ Clipped DEM saved to extracted_features/clark_clip.tif


AssertionError: ❌ Output geo bounds differ from requested LAT/LON by > ~2 pixels.
Requested (LL): (-115.1, 36.15, -115.05, 36.2)
Actual (LL):    (-115.10120669, 36.14919699, -115.04876322, 36.20080414)
Tolerance deg:  ~0.00002226

In [9]:
with rasterio.open("extracted_features/clark_clip.tif") as src:
    print("Pixel size (x, y):", src.res)


Pixel size (x, y): (1.0, 1.0)


In [ ]:
# road_slope_stats.py
import os, math
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union
import rasterio
from rasterio.transform import Affine
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
INPUT_DEM = "extracted_features/clark_clip.tif"
INPUT_ROADS = "extracted_features/small-nevada-roads.geojson"   # change if needed
OUTPUT_CSV = "extracted_features/road_slope_stats.csv"
OUTPUT_GEOJSON = "extracted_features/road_slope_roads.geojson"

SAMPLE_SPACING_M = 5.0     # sampling step along roads (meters)
THRESHOLD_PCT = 10.0       # “too steep” threshold (percent grade)
ID_FIELD = None            # existing road id column to keep; None -> auto "road_id"

# -----------------------------
# SLOPE (Horn 3x3) in % grade
# -----------------------------
def horn_slope_percent(elev, xres, yres):
    dzdx = (
        (elev[0:-2, 2:] + 2*elev[1:-1, 2:] + elev[2:, 2:]) -  
        (elev[0:-2, 0:-2] + 2*elev[1:-1, 0:-2] + elev[2:, 0:-2])
    ) / (8 * xres)
    dzdy = (
        (elev[2:, 0:-2] + 2*elev[2:, 1:-1] + elev[2:, 2:]) -
        (elev[0:-2, 0:-2] + 2*elev[0:-2, 1:-1] + elev[0:-2, 2:])
    ) / (8 * yres)
    return np.hypot(dzdx, dzdy) * 100.0

# -----------------------------
# Densify a LineString by spacing
# -----------------------------
def densify_line(line: LineString, spacing: float) -> np.ndarray:
    if line.length == 0:
        return np.empty((0, 2))
    n = max(1, int(math.ceil(line.length / spacing)))
    dists = np.linspace(0, line.length, n + 1)
    pts = [line.interpolate(d) for d in dists]
    return np.array([(p.x, p.y) for p in pts])

# -----------------------------
# Nearest-neighbour sampler for a 2D array using an Affine transform
# -----------------------------
def sample_array_nn(arr2d: np.ndarray, transform: Affine, xy_iter):
    inv = ~transform
    out = []
    h, w = arr2d.shape
    for x, y in xy_iter:
        colf, rowf = inv * (x, y)   # note: returns (col, row) as floats
        r = int(round(rowf))
        c = int(round(colf))
        if 0 <= r < h and 0 <= c < w:
            out.append(arr2d[r, c])
        else:
            out.append(np.nan)
    return out

# -----------------------------
# MAIN
# -----------------------------
def main():
    # 1) DEM → slope raster (%)
    with rasterio.open(INPUT_DEM) as src:
        T: Affine = src.transform
        crs = src.crs
        if not np.isclose(T.b, 0.0) or not np.isclose(T.d, 0.0):
            raise ValueError("DEM has rotation/skew; script assumes north-up pixels.")
        elev = src.read(1).astype("float32")
        nodata = src.nodata
        if nodata is not None:
            elev = np.where(elev == nodata, np.nan, elev)
        xres, yres = abs(T.a), abs(T.e)

        slope_core = horn_slope_percent(elev, xres, yres)
        slope = np.full(elev.shape, np.nan, dtype="float32")
        slope[1:-1, 1:-1] = slope_core
        dem_crs = crs

    # 2) Load roads & reproject to DEM CRS
    gdf = gpd.read_file(INPUT_ROADS)
    if gdf.empty:
        raise RuntimeError("No road features found.")
    if gdf.crs is None:
        raise RuntimeError("Roads layer has no CRS. Set it before running.")
    if gdf.crs != dem_crs:
        gdf = gdf.to_crs(dem_crs)

    if ID_FIELD is None or ID_FIELD not in gdf.columns:
        gdf["road_id"] = np.arange(1, len(gdf) + 1, dtype=int)
        id_field = "road_id"
    else:
        id_field = ID_FIELD

    # 3) Per-road sampling
    records = []
    steep_flags = []

    for i, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            records.append(dict(count=0, mean=np.nan, max=np.nan, p95=np.nan, steep_frac=np.nan))
            steep_flags.append(False)
            continue

        # Merge MultiLineString to single LineString (prefer longest)
        if geom.geom_type == "MultiLineString":
            merged = unary_union(list(geom.geoms))
            if merged.geom_type == "MultiLineString":
                geom = max(list(merged.geoms), key=lambda L: L.length)
            else:
                geom = merged

        if geom.geom_type != "LineString":
            records.append(dict(count=0, mean=np.nan, max=np.nan, p95=np.nan, steep_frac=np.nan))
            steep_flags.append(False)
            continue

        pts = densify_line(geom, SAMPLE_SPACING_M)
        if pts.shape[0] == 0:
            records.append(dict(count=0, mean=np.nan, max=np.nan, p95=np.nan, steep_frac=np.nan))
            steep_flags.append(False)
            continue

        vals = np.array(
            sample_array_nn(slope, T, [(float(x), float(y)) for x, y in pts]),
            dtype=float
        )
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            stats = dict(count=0, mean=np.nan, max=np.nan, p95=np.nan, steep_frac=np.nan)
            steep = False
        else:
            stats = dict(
                count=int(vals.size),
                mean=float(np.nanmean(vals)),
                max=float(np.nanmax(vals)),
                p95=float(np.nanpercentile(vals, 95)),
                steep_frac=float(np.mean(vals >= THRESHOLD_PCT))
            )
            steep = bool(np.any(vals >= THRESHOLD_PCT))

        records.append(stats)
        steep_flags.append(steep)

    # 4) Outputs
    df = pd.DataFrame.from_records(
        [
            {
                id_field: gdf.iloc[i][id_field],
                "length_m": float(gdf.iloc[i].geometry.length) if gdf.iloc[i].geometry else np.nan,
                "sample_count": rec["count"],
                "slope_mean_pct": rec["mean"],
                "slope_max_pct": rec["max"],
                "slope_p95_pct": rec["p95"],
                "steep_frac": rec["steep_frac"],
                "too_steep_flag": steep_flags[i],
            }
            for i, rec in enumerate(records)
        ]
    )

    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False)

    gdf_out = gdf[[id_field, "geometry"]].copy()
    gdf_out = gdf_out.merge(df, on=id_field, how="left")
    os.makedirs(os.path.dirname(OUTPUT_GEOJSON), exist_ok=True)
    gdf_out.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

    print(f"✅ Wrote per-road slope stats: {OUTPUT_CSV}")
    print(f"✅ Wrote roads with attributes: {OUTPUT_GEOJSON}")
    print(f"ℹ️ Threshold: {THRESHOLD_PCT:.1f}% | Spacing: {SAMPLE_SPACING_M} m | DEM CRS: {dem_crs.to_string() if dem_crs else None}")

if __name__ == "__main__":
    main()


✅ Wrote per-road slope stats: extracted_features/road_slope_stats.csv
✅ Wrote roads with attributes: extracted_features/road_slope_roads.geojson
ℹ️ Threshold: 10.0% | Spacing: 5.0 m | DEM CRS: EPSG:26911


In [2]:
# requirements: geopandas, shapely>=2, rasterio, numpy, tqdm (optional)
import geopandas as gpd
import numpy as np
import rasterio
from rasterio import features
from rasterio.sample import sample_gen
from shapely.geometry import LineString
from shapely.ops import substring
from tqdm import tqdm
import os
# ---------- CONFIG ----------
ROADS_PATH = "extracted_features/small-nevada-roads.geojson"   # your input
DEM_PATH   = "extracted_features/clark_clip.tif"             # <-- set this
OUT_PATH   = "extracted_features/small-nevada-roads_with_slope.geojson"
# densification step (meters) for avg/max slope; set None to skip
STEP_M = 10.0
# --------------------------------

# remove this import (not needed)
# from rasterio.sample import sample_gen

def sample_dem_at_coords(ds, xs, ys, nodata=None):
    """Return DEM values at arrays of x,y; nodata -> np.nan."""
    coords = list(zip(xs, ys))                 # [(x,y), ...]
    vals = list(ds.sample(coords))             # sample via dataset method
    arr = np.array([v[0] if len(v) else np.nan for v in vals], dtype=float)
    if nodata is not None:
        arr = np.where(np.isclose(arr, nodata), np.nan, arr)
    return arr


def endpoints(line: LineString):
    if line.is_empty or line.length == 0:
        return None, None
    x0, y0 = line.coords[0]
    x1, y1 = line.coords[-1]
    return (x0, y0), (x1, y1)

def points_along(line: LineString, step: float):
    """Return coords every `step` meters along the line (including endpoints)."""
    L = line.length
    if L == 0:
        return [line.coords[0]]
    ds = np.arange(0.0, L, step)
    if not np.isclose(ds[-1], L):
        ds = np.append(ds, L)
    coords = []
    for d in ds:
        seg = substring(line, d, d, normalized=False)
        # substring at same start/stop returns a point-like; get its first coordinate
        coords.append(seg.coords[0])
    return coords

def main():
    # open DEM
    with rasterio.open(DEM_PATH) as dem:
        dem_crs = dem.crs
        dem_nodata = dem.nodata
        dem_units = "meters"  # USGS 3DEP DEMs are meters

        # read roads and reproject to DEM CRS (assumed meters -> length correct)
        gdf = gpd.read_file(ROADS_PATH)
        if gdf.crs is None:
            raise ValueError("Input roads have no CRS. Please set the correct CRS before running.")
        if gdf.crs != dem_crs:
            gdf = gdf.to_crs(dem_crs)

        # ensure LineString/MultiLineString
        gdf = gdf[~gdf.geometry.is_empty].copy()
        gdf["length_m"] = gdf.length

        # containers
        elev_start = np.full(len(gdf), np.nan)
        elev_end   = np.full(len(gdf), np.nan)
        grade_pct  = np.full(len(gdf), np.nan)
        slope_deg  = np.full(len(gdf), np.nan)

        avg_grade_pct = np.full(len(gdf), np.nan)
        max_grade_pct = np.full(len(gdf), np.nan)
        n_samples     = np.zeros(len(gdf), dtype=int)

        # iterate
        for i, geom in tqdm(list(enumerate(gdf.geometry)), total=len(gdf)):
            if geom.is_empty:
                continue

            # dissolve MultiLineString to single LineString by chaining parts if needed
            if geom.geom_type == "MultiLineString":
                parts = [ls for ls in geom.geoms if ls.length > 0]
                if not parts:
                    continue
                # connect parts by order; for slope we just need endpoints along total path
                geom = LineString([pt for ls in parts for pt in ls.coords])

            if geom.length == 0:
                continue

            # start/end elevations
            (x0, y0), (x1, y1) = endpoints(geom)
            with rasterio.open(DEM_PATH) as dem_ds:
                zs = sample_dem_at_coords(dem_ds, np.array([x0, x1]), np.array([y0, y1]), dem_nodata)
            z0, z1 = zs[0], zs[1]
            elev_start[i] = z0
            elev_end[i]   = z1

            L = geom.length  # meters
            if np.isfinite(z0) and np.isfinite(z1) and L > 0:
                dz = (z1 - z0)
                grade_pct[i] = (dz / L) * 100.0
                slope_deg[i] = np.degrees(np.arctan(dz / L))

            # densified sampling for avg/max grade
            if STEP_M and STEP_M > 0:
                coords = points_along(geom, STEP_M)
                xs = np.array([c[0] for c in coords], dtype=float)
                ys = np.array([c[1] for c in coords], dtype=float)
                with rasterio.open(DEM_PATH) as dem_ds:
                    zs = sample_dem_at_coords(dem_ds, xs, ys, dem_nodata)

                # pairwise slopes between consecutive samples
                # distance is ~STEP_M except the last short segment
                if len(zs) >= 2:
                    dzs = np.diff(zs)
                    # segment lengths by linear referencing
                    # replace the old seg_lengths computation with:
                    seg_lengths = np.array([
                        LineString([coords[j], coords[j+1]]).length
                        for j in range(len(coords)-1)
                    ], dtype=float)

                    # guard against zeros/NaNs
                    mask = (seg_lengths > 0) & np.isfinite(dzs)
                    if mask.any():
                        grades = (dzs[mask] / seg_lengths[mask]) * 100.0
                        avg_grade_pct[i] = np.nanmean(grades) if grades.size else np.nan
                        max_grade_pct[i] = np.nanmax(np.abs(grades)) if grades.size else np.nan
                        n_samples[i] = len(zs)

        # write out
        gdf["elev_start_m"] = elev_start
        gdf["elev_end_m"]   = elev_end
        gdf["grade_pct"]    = grade_pct
        gdf["slope_deg"]    = slope_deg
        gdf["avg_grade_pct_step"] = avg_grade_pct
        gdf["max_abs_grade_pct_step"] = max_grade_pct
        gdf["n_dem_samples"] = n_samples

        # save
        os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
        gdf.to_file(OUT_PATH, driver="GeoJSON")
        print(f"✅ Wrote: {OUT_PATH}")

if __name__ == "__main__":
    main()


100%|██████████| 4335/4335 [00:23<00:00, 187.05it/s]


✅ Wrote: extracted_features/small-nevada-roads_with_slope.geojson
